# Sequential Goal Forecasting With Geometry-Aware Sequences

This notebook adds a sequence-oriented extension on top of the existing checkpoint panel.

It implements two aligned sequence views:

- `minute_hybrid`: 15-minute ordered bins before each checkpoint
- `possession_sequence`: strict possession summaries from run and shot events, with unmatched pressure retained as a residual summary

Outputs are exported to `artifacts/sequential_goal_forecasting_geometry/`.

In [1]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / 'data').exists())
DATA_DIR = ROOT / 'data'
ARTIFACT_DIR = ROOT / 'artifacts' / 'sequential_goal_forecasting_geometry'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
LOOKBACK_MINUTES = 15
FORWARD_ANGLE_DEG = 30
LATERAL_ANGLE_DEG = 120
PRESSURE_RICH_THRESHOLD = 2
LATE_BURST_THRESHOLD = 2

PERIOD_OFFSETS = {
    'half_1': 0,
    'half_2': 45,
    'extra_time_1': 90,
    'extra_time_2': 105,
}
CHECKPOINT_TO_CONT_MIN = {
    'H1_15': 15,
    'H1_30': 30,
    'H1_45': 45,
    'H2_15': 60,
    'H2_30': 75,
    'H2_45': 90,
    'ET1_15': 105,
}
CATEGORICAL_BASE_FEATURES = ['checkpoint', 'position', 'formation']
BASE_FEATURE_COLUMNS = [
    'checkpoint',
    'checkpoint_min',
    'position',
    'is_home',
    'formation',
    'minute_in',
    'minutes_available_before_checkpoint',
    'last15_sprints',
    'last15_hsr',
    'last15_distance',
    'last15_mean_max_speed',
    'last15_peak_speed',
    'last15_shots',
    'last15_shots_on_target',
    'last15_shots_under_press',
    'last15_shots_top_third',
    'cumul_sprints',
    'cumul_hsr',
    'cumul_distance',
    'cumul_mean_max_speed',
    'cumul_peak_speed',
    'cumul_shots',
    'cumul_shots_on_target',
    'cumul_shots_under_press',
    'cumul_shots_top_third',
]
TRIMMED_SEQ15M_FEATURES = [
    'seq15m_pressure_total',
    'seq15m_pressure_turnover_total',
    'seq15m_pressure_angle_availability_rate',
    'seq15m_angle_concentration',
    'seq15m_lateral_angle_share',
    'seq15m_pressured_minutes_with_runs_share',
    'seq15m_late_burst_count',
    'seq15m_active_minute_share',
    'seq15m_recency_weighted_pressure_total',
    'seq15m_run_intensity_slope',
]
TRIMMED_SEQPOS_FEATURES = [
    'seqpos_possession_count',
    'seqpos_mean_runs_per_possession',
    'seqpos_mean_run_distance_per_possession',
    'seqpos_max_run_distance_per_possession',
    'seqpos_mean_possession_span',
    'seqpos_linked_pressure_total',
    'seqpos_possessions_with_linked_pressure_share',
    'seqpos_unmatched_pressure_share',
    'seqpos_unmatched_angle_availability_rate',
    'seqpos_strict_link_rate',
]
LEAN_SEQ15M_FEATURES = [
    'seq15m_pressure_total',
    'seq15m_pressure_turnover_total',
    'seq15m_angle_concentration',
    'seq15m_active_minute_share',
    'seq15m_late_burst_count',
]
LEAN_SEQPOS_FEATURES = [
    'seqpos_possession_count',
    'seqpos_mean_runs_per_possession',
    'seqpos_mean_run_distance_per_possession',
    'seqpos_linked_pressure_total',
    'seqpos_unmatched_pressure_share',
]

/Users/jan/.matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /var/folders/pq/224wz8116dq05pr87bp6rkb80000gn/T/matplotlib-i0izgxiz because there was an issue with the default path (/Users/jan/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Matplotlib is building the font cache; this may take a moment.


In [2]:
def add_event_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['event_cont_min'] = out['period'].map(PERIOD_OFFSETS) + pd.to_numeric(out['minute'], errors='coerce')
    return out


def safe_rate(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def circular_mean_deg(values: np.ndarray) -> float:
    if values.size == 0:
        return 0.0
    radians = np.deg2rad(values)
    return math.degrees(math.atan2(np.sin(radians).mean(), np.cos(radians).mean()))


def circular_diff_deg(a_deg: float, b_deg: float) -> float:
    diff = (a_deg - b_deg + 180.0) % 360.0 - 180.0
    return abs(diff)


def longest_true_streak(mask: pd.Series) -> int:
    best = 0
    current = 0
    for value in mask.astype(bool).tolist():
        current = current + 1 if value else 0
        best = max(best, current)
    return int(best)


def linear_slope(values: np.ndarray) -> float:
    if values.size <= 1:
        return 0.0
    x = np.arange(values.size, dtype=float)
    x = x - x.mean()
    y = values.astype(float) - values.astype(float).mean()
    denominator = np.square(x).sum()
    if denominator == 0:
        return 0.0
    return float((x * y).sum() / denominator)


def build_preprocessor(feature_columns: list[str]) -> ColumnTransformer:
    categorical_columns = [column for column in feature_columns if column in CATEGORICAL_BASE_FEATURES]
    numeric_columns = [column for column in feature_columns if column not in categorical_columns]
    return ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
                        ('scaler', StandardScaler()),
                    ]
                ),
                numeric_columns,
            ),
            (
                'cat',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='most_frequent')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore')),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )


def score_predictions(y_true: np.ndarray, pred_prob: np.ndarray) -> dict[str, float]:
    pred_label = (pred_prob >= 0.5).astype(int)
    metrics = {
        'balanced_accuracy': balanced_accuracy_score(y_true, pred_label),
        'pr_auc': average_precision_score(y_true, pred_prob),
        'brier_score': brier_score_loss(y_true, pred_prob),
    }
    metrics['roc_auc'] = roc_auc_score(y_true, pred_prob) if len(np.unique(y_true)) > 1 else np.nan
    return metrics


def subgroup_metric_table(predictions: pd.DataFrame, group_column: str) -> pd.DataFrame:
    rows = []
    for group_value, subset in predictions.groupby(group_column, dropna=False):
        row = {
            'group_type': group_column,
            'group_value': group_value,
            'rows': int(len(subset)),
            'positive_rate': float(subset['scored_after'].mean()),
            'balanced_accuracy': balanced_accuracy_score(
                subset['scored_after'],
                (subset['pred_prob'] >= 0.5).astype(int),
            ),
            'pr_auc': average_precision_score(subset['scored_after'], subset['pred_prob']),
            'brier_score': brier_score_loss(subset['scored_after'], subset['pred_prob']),
        }
        row['roc_auc'] = roc_auc_score(subset['scored_after'], subset['pred_prob']) if subset['scored_after'].nunique() > 1 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['group_type', 'rows'], ascending=[True, False]).reset_index(drop=True)


def evaluate_feature_set(
    development_df: pd.DataFrame,
    feature_columns: list[str],
    model_name: str,
    return_oof: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    if model_name == 'logreg':
        model = LogisticRegression(
            max_iter=4000,
            class_weight='balanced',
            solver='liblinear',
            C=0.25,
            random_state=RANDOM_STATE,
        )
    elif model_name == 'rf':
        model = RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')

    folds = sorted(development_df['fold'].dropna().unique())
    fold_rows = []
    oof_rows = []
    for fold in folds:
        train_df = development_df[development_df['fold'] != fold].copy()
        valid_df = development_df[development_df['fold'] == fold].copy()
        pipeline = Pipeline(
            steps=[
                ('prep', build_preprocessor(feature_columns)),
                ('model', model),
            ]
        )
        pipeline.fit(train_df[feature_columns], train_df['scored_after'].astype(int))
        pred_prob = pipeline.predict_proba(valid_df[feature_columns])[:, 1]
        fold_metrics = score_predictions(valid_df['scored_after'].to_numpy(), pred_prob)
        fold_rows.append({'fold': fold, **fold_metrics})

        if return_oof:
            oof_part = valid_df[
                ['player_appearance_id', 'fixture_id', 'checkpoint', 'position', 'scored_after']
            ].copy()
            oof_part['fold'] = fold
            oof_part['pred_prob'] = pred_prob
            oof_rows.append(oof_part)

    fold_metrics_df = pd.DataFrame(fold_rows)
    oof_df = pd.concat(oof_rows, ignore_index=True) if return_oof else None
    return fold_metrics_df, oof_df

In [3]:
base = pd.read_csv(DATA_DIR / 'players_quarters_final.csv')
pressure = pd.read_csv(DATA_DIR / 'player_appearance_behaviour_under_pressure.csv')
runs = pd.read_csv(DATA_DIR / 'player_appearance_run.csv')
shots = pd.read_csv(DATA_DIR / 'player_appearance_shot_limited.csv')
fixture_assignments = pd.read_csv(DATA_DIR / 'splits' / 'fixture_assignments.csv')

base['checkpoint_cont_min'] = base['checkpoint'].map(CHECKPOINT_TO_CONT_MIN)
base['minutes_available_before_checkpoint'] = base['checkpoint_cont_min'] - base['minute_in'] + 1
base = base.merge(fixture_assignments, on='fixture_id', how='left', validate='many_to_one')

pressure = add_event_time(pressure)
pressure['pass_angle_observed'] = pressure['pass_angle'].notna().astype(int)
pressure['pass_angle_rad'] = np.deg2rad(pressure['pass_angle'])
pressure['pass_angle_sin'] = np.sin(pressure['pass_angle_rad']).fillna(0.0)
pressure['pass_angle_cos'] = np.cos(pressure['pass_angle_rad']).fillna(0.0)
pressure['pass_angle_abs'] = pressure['pass_angle'].abs()
pressure['forward_angle_flag'] = pressure['pass_angle_abs'].le(FORWARD_ANGLE_DEG).fillna(False).astype(int)
pressure['lateral_angle_flag'] = (
    pressure['pass_angle_abs'].gt(FORWARD_ANGLE_DEG).fillna(False)
    & pressure['pass_angle_abs'].le(LATERAL_ANGLE_DEG).fillna(False)
).astype(int)
pressure['backward_angle_flag'] = pressure['pass_angle_abs'].gt(LATERAL_ANGLE_DEG).fillna(False).astype(int)
pressure['turnover_flag'] = pressure['press_induced_outcome'].eq('turnover').astype(int)

runs = add_event_time(runs)

shots = add_event_time(shots)
shots = shots[shots['own_goal_player_appearance_id'].isna()].copy()
shots['under_pressure_flag'] = shots['under_pressure'].astype(int)
shots['top_stage_flag'] = shots['stage'].eq('top').astype(int)

strict_possession_map = (
    runs.groupby(['player_appearance_id', 'period', 'minute'], dropna=False)
    .agg(possession_nunique=('possession', 'nunique'), strict_possession=('possession', 'first'))
    .reset_index()
)
strict_possession_map.loc[
    strict_possession_map['possession_nunique'] != 1,
    'strict_possession',
] = np.nan

pressure = pressure.merge(
    strict_possession_map[
        ['player_appearance_id', 'period', 'minute', 'strict_possession', 'possession_nunique']
    ],
    on=['player_appearance_id', 'period', 'minute'],
    how='left',
)
pressure.rename(columns={'strict_possession': 'linked_possession'}, inplace=True)
pressure['linked_pressure_flag'] = pressure['linked_possession'].notna().astype(int)

display(Markdown('### Loaded Tables'))
display(
    pd.DataFrame(
        [
            {'table': 'players_quarters_final', 'rows': len(base), 'columns': base.shape[1]},
            {'table': 'player_appearance_behaviour_under_pressure', 'rows': len(pressure), 'columns': pressure.shape[1]},
            {'table': 'player_appearance_run', 'rows': len(runs), 'columns': runs.shape[1]},
            {'table': 'player_appearance_shot_limited', 'rows': len(shots), 'columns': shots.shape[1]},
        ]
    )
)

### Loaded Tables

,table,rows,columns
0,players_quarters_final,3486,37
1,player_appearance_behaviour_under_pressure,12185,23
2,player_appearance_run,35133,11
3,player_appearance_shot_limited,779,15


In [4]:
pressure_groups = {
    player_appearance_id: frame.sort_values(['event_cont_min', 'id']).reset_index(drop=True)
    for player_appearance_id, frame in pressure.groupby('player_appearance_id')
}
run_groups = {
    player_appearance_id: frame.sort_values(['event_cont_min', 'id']).reset_index(drop=True)
    for player_appearance_id, frame in runs.groupby('player_appearance_id')
}
shot_groups = {
    player_appearance_id: frame.sort_values(['event_cont_min', 'id']).reset_index(drop=True)
    for player_appearance_id, frame in shots.groupby('player_appearance_id')
}

sequence_rows = []
minute_sequence_rows = []
possession_sequence_rows = []

for row in base.itertuples(index=False):
    checkpoint_time = int(row.checkpoint_cont_min)
    history_start = max(1, checkpoint_time - LOOKBACK_MINUTES + 1)
    player_appearance_id = int(row.player_appearance_id)

    player_pressure = pressure_groups.get(player_appearance_id)
    player_runs = run_groups.get(player_appearance_id)
    player_shots = shot_groups.get(player_appearance_id)

    pressure_last15 = (
        player_pressure[
            (player_pressure['event_cont_min'] >= history_start)
            & (player_pressure['event_cont_min'] <= checkpoint_time)
        ].copy()
        if player_pressure is not None
        else pd.DataFrame(columns=pressure.columns)
    )
    runs_last15 = (
        player_runs[
            (player_runs['event_cont_min'] >= history_start)
            & (player_runs['event_cont_min'] <= checkpoint_time)
        ].copy()
        if player_runs is not None
        else pd.DataFrame(columns=runs.columns)
    )
    shots_last15 = (
        player_shots[
            (player_shots['event_cont_min'] >= history_start)
            & (player_shots['event_cont_min'] <= checkpoint_time)
        ].copy()
        if player_shots is not None
        else pd.DataFrame(columns=shots.columns)
    )

    pressure_pre = (
        player_pressure[player_pressure['event_cont_min'] <= checkpoint_time].copy()
        if player_pressure is not None
        else pd.DataFrame(columns=pressure.columns)
    )
    runs_pre = (
        player_runs[player_runs['event_cont_min'] <= checkpoint_time].copy()
        if player_runs is not None
        else pd.DataFrame(columns=runs.columns)
    )
    shots_pre = (
        player_shots[player_shots['event_cont_min'] <= checkpoint_time].copy()
        if player_shots is not None
        else pd.DataFrame(columns=shots.columns)
    )

    minute_rows = []
    for minute in range(history_start, checkpoint_time + 1):
        pressure_min = pressure_last15[pressure_last15['event_cont_min'] == minute]
        runs_min = runs_last15[runs_last15['event_cont_min'] == minute]
        shots_min = shots_last15[shots_last15['event_cont_min'] == minute]

        combined_possessions = pd.concat(
            [runs_min[['possession']], shots_min[['possession']]],
            ignore_index=True,
        )
        possession_transition_count = (
            max(int(combined_possessions['possession'].nunique()) - 1, 0)
            if not combined_possessions.empty
            else 0
        )

        minute_row = {
            'player_appearance_id': player_appearance_id,
            'checkpoint': row.checkpoint,
            'fixture_id': row.fixture_id,
            'checkpoint_cont_min': checkpoint_time,
            'minute_bin_cont_min': minute,
            'minute_bin_idx': minute - history_start + 1,
            'pressure_count': int(len(pressure_min)),
            'pressure_turnover_count': int(pressure_min['turnover_flag'].sum()) if not pressure_min.empty else 0,
            'pressure_angle_observed_count': int(pressure_min['pass_angle_observed'].sum()) if not pressure_min.empty else 0,
            'pressure_mean_sin': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'pass_angle_sin'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_mean_cos': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'pass_angle_cos'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_forward_share': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'forward_angle_flag'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_lateral_share': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'lateral_angle_flag'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_forward_count': int(pressure_min['forward_angle_flag'].sum()) if not pressure_min.empty else 0,
            'run_count': int(len(runs_min)),
            'run_distance_sum': float(runs_min['distance'].sum()) if not runs_min.empty else 0.0,
            'run_mean_speed': float(runs_min['max_speed'].mean()) if not runs_min.empty else 0.0,
            'run_max_speed': float(runs_min['max_speed'].max()) if not runs_min.empty else 0.0,
            'run_top_share': float(runs_min['stage'].eq('top').mean()) if not runs_min.empty else 0.0,
            'run_middle_share': float(runs_min['stage'].eq('middle').mean()) if not runs_min.empty else 0.0,
            'run_bottom_share': float(runs_min['stage'].eq('bottom').mean()) if not runs_min.empty else 0.0,
            'shot_count': int(len(shots_min)),
            'shot_under_pressure_count': int(shots_min['under_pressure_flag'].sum()) if not shots_min.empty else 0,
            'shot_top_stage_count': int(shots_min['top_stage_flag'].sum()) if not shots_min.empty else 0,
            'possession_transition_count': possession_transition_count,
        }
        minute_rows.append(minute_row)
        minute_sequence_rows.append(minute_row)

    minute_df = pd.DataFrame(minute_rows)
    activity_total = minute_df[['pressure_count', 'run_count', 'shot_count']].sum(axis=1)
    pressured_mask = minute_df['pressure_count'] > 0
    pressure_rich_mask = minute_df['pressure_count'] >= PRESSURE_RICH_THRESHOLD
    forward_pressure_mask = minute_df['pressure_forward_count'] > 0
    pressure_or_previous_mask = pressured_mask | pressured_mask.shift(fill_value=False)
    forward_pressure_or_previous_mask = forward_pressure_mask | forward_pressure_mask.shift(fill_value=False)
    observed_angles = pressure_last15.loc[pressure_last15['pass_angle'].notna(), 'pass_angle'].to_numpy(dtype=float)

    if observed_angles.size:
        radians = np.deg2rad(observed_angles)
        angle_concentration = float(
            np.sqrt(np.square(np.sin(radians).sum()) + np.square(np.cos(radians).sum())) / observed_angles.size
        )
        early_angles = pressure_last15.loc[
            (pressure_last15['event_cont_min'] < checkpoint_time - LOOKBACK_MINUTES / 2 + 1)
            & pressure_last15['pass_angle'].notna(),
            'pass_angle',
        ].to_numpy(dtype=float)
        late_angles = pressure_last15.loc[
            (pressure_last15['event_cont_min'] >= checkpoint_time - LOOKBACK_MINUTES / 2 + 1)
            & pressure_last15['pass_angle'].notna(),
            'pass_angle',
        ].to_numpy(dtype=float)
        directional_change_deg = (
            circular_diff_deg(circular_mean_deg(late_angles), circular_mean_deg(early_angles))
            if early_angles.size and late_angles.size
            else 0.0
        )
    else:
        angle_concentration = 0.0
        directional_change_deg = 0.0

    rich_following_run_distance = minute_df.loc[
        pressure_rich_mask.shift(fill_value=False),
        'run_distance_sum',
    ]
    forward_shot_counts = minute_df.loc[forward_pressure_or_previous_mask, 'shot_count']
    forward_up_shot_counts = minute_df.loc[
        forward_pressure_or_previous_mask,
        'shot_under_pressure_count',
    ]
    weights = np.linspace(1.0 / len(minute_df), 1.0, len(minute_df))

    linked_pressure_pre = pressure_pre[pressure_pre['linked_pressure_flag'] == 1].copy()
    unmatched_pressure_pre = pressure_pre[pressure_pre['linked_pressure_flag'] == 0].copy()
    total_pressure_pre = int(len(pressure_pre))

    possession_parts = []
    if not runs_pre.empty:
        runs_part = runs_pre[['event_cont_min', 'possession', 'distance']].copy()
        runs_part['run_count'] = 1
        runs_part['shot_count'] = 0
        runs_part['linked_pressure_count'] = 0
        runs_part['forward_pressure_count'] = 0
        possession_parts.append(runs_part)
    if not shots_pre.empty:
        shots_part = shots_pre[['event_cont_min', 'possession']].copy()
        shots_part['distance'] = 0.0
        shots_part['run_count'] = 0
        shots_part['shot_count'] = 1
        shots_part['linked_pressure_count'] = 0
        shots_part['forward_pressure_count'] = 0
        possession_parts.append(shots_part)
    if not linked_pressure_pre.empty:
        pressure_part = linked_pressure_pre[['event_cont_min', 'linked_possession', 'forward_angle_flag']].copy()
        pressure_part.rename(columns={'linked_possession': 'possession'}, inplace=True)
        pressure_part['distance'] = 0.0
        pressure_part['run_count'] = 0
        pressure_part['shot_count'] = 0
        pressure_part['linked_pressure_count'] = 1
        pressure_part['forward_pressure_count'] = pressure_part['forward_angle_flag']
        possession_parts.append(pressure_part.drop(columns=['forward_angle_flag']))

    if possession_parts:
        possession_df = pd.concat(possession_parts, ignore_index=True)
        possession_summary = (
            possession_df.groupby('possession', dropna=True)
            .agg(
                possession_min=('event_cont_min', 'min'),
                possession_max=('event_cont_min', 'max'),
                run_count=('run_count', 'sum'),
                run_distance_sum=('distance', 'sum'),
                shot_count=('shot_count', 'sum'),
                linked_pressure_count=('linked_pressure_count', 'sum'),
                forward_pressure_count=('forward_pressure_count', 'sum'),
            )
            .reset_index()
        )
        possession_summary['event_span'] = (
            possession_summary['possession_max'] - possession_summary['possession_min'] + 1
        )
    else:
        possession_summary = pd.DataFrame(
            columns=[
                'possession',
                'possession_min',
                'possession_max',
                'run_count',
                'run_distance_sum',
                'shot_count',
                'linked_pressure_count',
                'forward_pressure_count',
                'event_span',
            ]
        )

    if not possession_summary.empty:
        possession_export = possession_summary.copy()
        possession_export.insert(0, 'player_appearance_id', player_appearance_id)
        possession_export.insert(1, 'checkpoint', row.checkpoint)
        possession_export.insert(2, 'fixture_id', row.fixture_id)
        possession_export.insert(3, 'checkpoint_cont_min', checkpoint_time)
        possession_sequence_rows.extend(possession_export.to_dict(orient='records'))

    sequence_rows.append(
        {
            'player_appearance_id': player_appearance_id,
            'checkpoint': row.checkpoint,
            'fixture_id': row.fixture_id,
            'checkpoint_cont_min': checkpoint_time,
            'seq15m_pressure_total': int(minute_df['pressure_count'].sum()),
            'seq15m_pressure_turnover_total': int(minute_df['pressure_turnover_count'].sum()),
            'seq15m_run_total': int(minute_df['run_count'].sum()),
            'seq15m_run_distance_total': float(minute_df['run_distance_sum'].sum()),
            'seq15m_shot_total': int(minute_df['shot_count'].sum()),
            'seq15m_shot_under_pressure_total': int(minute_df['shot_under_pressure_count'].sum()),
            'seq15m_pressure_angle_availability_rate': safe_rate(
                int(minute_df['pressure_angle_observed_count'].sum()),
                int(minute_df['pressure_count'].sum()),
            ),
            'seq15m_forward_angle_share': safe_rate(
                int(pressure_last15['forward_angle_flag'].sum()),
                int(pressure_last15['pass_angle_observed'].sum()),
            ),
            'seq15m_lateral_angle_share': safe_rate(
                int(pressure_last15['lateral_angle_flag'].sum()),
                int(pressure_last15['pass_angle_observed'].sum()),
            ),
            'seq15m_backward_angle_share': safe_rate(
                int(pressure_last15['backward_angle_flag'].sum()),
                int(pressure_last15['pass_angle_observed'].sum()),
            ),
            'seq15m_angle_concentration': angle_concentration,
            'seq15m_angle_dispersion': 1.0 - angle_concentration if observed_angles.size else 0.0,
            'seq15m_directional_change_deg': float(directional_change_deg),
            'seq15m_pressured_minutes_with_runs_share': safe_rate(
                int(((minute_df['pressure_count'] > 0) & (minute_df['run_count'] > 0)).sum()),
                int((minute_df['pressure_count'] > 0).sum()),
            ),
            'seq15m_run_distance_after_pressure_rich_mean': float(rich_following_run_distance.mean())
            if len(rich_following_run_distance)
            else 0.0,
            'seq15m_shots_same_or_next_min_after_pressure': int(
                minute_df.loc[pressure_or_previous_mask, 'shot_count'].sum()
            ),
            'seq15m_under_pressure_shot_rate_after_forward_pressure': safe_rate(
                int(forward_up_shot_counts.sum()),
                int(forward_shot_counts.sum()),
            ),
            'seq15m_late_burst_count': int(activity_total.tail(5).ge(LATE_BURST_THRESHOLD).sum()),
            'seq15m_longest_active_streak': longest_true_streak(activity_total > 0),
            'seq15m_active_minute_share': float((activity_total > 0).mean()),
            'seq15m_recency_weighted_pressure_total': float(
                np.dot(minute_df['pressure_count'].to_numpy(dtype=float), weights)
            ),
            'seq15m_recency_weighted_run_total': float(
                np.dot(minute_df['run_count'].to_numpy(dtype=float), weights)
            ),
            'seq15m_recency_weighted_shot_total': float(
                np.dot(minute_df['shot_count'].to_numpy(dtype=float), weights)
            ),
            'seq15m_pressure_density_slope': linear_slope(
                minute_df['pressure_count'].to_numpy(dtype=float)
            ),
            'seq15m_run_intensity_slope': linear_slope(
                minute_df['run_distance_sum'].to_numpy(dtype=float)
            ),
            'seq15m_possession_transition_total': int(minute_df['possession_transition_count'].sum()),
            'seqpos_possession_count': int(len(possession_summary)),
            'seqpos_possessions_with_shots_share': float(
                (possession_summary['shot_count'] > 0).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_possessions_with_linked_pressure_share': float(
                (possession_summary['linked_pressure_count'] > 0).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_mean_runs_per_possession': float(possession_summary['run_count'].mean())
            if not possession_summary.empty
            else 0.0,
            'seqpos_mean_run_distance_per_possession': float(
                possession_summary['run_distance_sum'].mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_max_run_distance_per_possession': float(
                possession_summary['run_distance_sum'].max()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_mean_shots_per_possession': float(possession_summary['shot_count'].mean())
            if not possession_summary.empty
            else 0.0,
            'seqpos_pressure_shot_escalation_share': float(
                ((possession_summary['linked_pressure_count'] > 0) & (possession_summary['shot_count'] > 0)).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_forward_pressure_shot_share': float(
                ((possession_summary['forward_pressure_count'] > 0) & (possession_summary['shot_count'] > 0)).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_linked_pressure_total': int(possession_summary['linked_pressure_count'].sum())
            if not possession_summary.empty
            else 0,
            'seqpos_mean_possession_span': float(possession_summary['event_span'].mean())
            if not possession_summary.empty
            else 0.0,
            'seqpos_unmatched_pressure_count': int(len(unmatched_pressure_pre)),
            'seqpos_unmatched_pressure_share': safe_rate(int(len(unmatched_pressure_pre)), total_pressure_pre),
            'seqpos_unmatched_turnover_share': float(unmatched_pressure_pre['turnover_flag'].mean())
            if not unmatched_pressure_pre.empty
            else 0.0,
            'seqpos_unmatched_angle_availability_rate': float(
                unmatched_pressure_pre['pass_angle_observed'].mean()
            )
            if not unmatched_pressure_pre.empty
            else 0.0,
            'seqpos_strict_link_rate': safe_rate(int(linked_pressure_pre.shape[0]), total_pressure_pre),
            '_seq15m_max_event_cont_min': max(
                [
                    pressure_last15['event_cont_min'].max() if not pressure_last15.empty else 0,
                    runs_last15['event_cont_min'].max() if not runs_last15.empty else 0,
                    shots_last15['event_cont_min'].max() if not shots_last15.empty else 0,
                ]
            ),
            '_seqpos_max_event_cont_min': max(
                [
                    pressure_pre['event_cont_min'].max() if not pressure_pre.empty else 0,
                    runs_pre['event_cont_min'].max() if not runs_pre.empty else 0,
                    shots_pre['event_cont_min'].max() if not shots_pre.empty else 0,
                ]
            ),
        }
    )

sequence_feature_table = pd.DataFrame(sequence_rows).sort_values(
    ['fixture_id', 'player_appearance_id', 'checkpoint_cont_min']
).reset_index(drop=True)
minute_sequence_long = pd.DataFrame(minute_sequence_rows).sort_values(
    ['fixture_id', 'player_appearance_id', 'checkpoint_cont_min', 'minute_bin_cont_min']
).reset_index(drop=True)
possession_sequence_long = pd.DataFrame(possession_sequence_rows).sort_values(
    ['fixture_id', 'player_appearance_id', 'checkpoint_cont_min', 'possession_min', 'possession']
).reset_index(drop=True)

display(Markdown('### Sequence Feature Table Shape'))
display(
    pd.DataFrame(
        [
            {'output': 'sequence_feature_table', 'rows': len(sequence_feature_table), 'columns': sequence_feature_table.shape[1]},
            {'output': 'minute_sequence_long', 'rows': len(minute_sequence_long), 'columns': minute_sequence_long.shape[1]},
            {'output': 'possession_sequence_long', 'rows': len(possession_sequence_long), 'columns': possession_sequence_long.shape[1]},
        ]
    )
)
display(sequence_feature_table.head())

### Sequence Feature Table Shape

,output,rows,columns
0,sequence_feature_table,3486,48
1,minute_sequence_long,52290,25
2,possession_sequence_long,6448,13


,player_appearance_id,checkpoint,fixture_id,checkpoint_cont_min,seq15m_pressure_total,seq15m_pressure_turnover_total,seq15m_run_total,seq15m_run_distance_total,seq15m_shot_total,seq15m_shot_under_pressure_total,...,seqpos_forward_pressure_shot_share,seqpos_linked_pressure_total,seqpos_mean_possession_span,seqpos_unmatched_pressure_count,seqpos_unmatched_pressure_share,seqpos_unmatched_turnover_share,seqpos_unmatched_angle_availability_rate,seqpos_strict_link_rate,_seq15m_max_event_cont_min,_seqpos_max_event_cont_min
0,39421,H1_15,1159,15,0,0,0,0.000000,0,0,...,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,0
1,39421,H1_30,1159,30,0,0,0,0.000000,0,0,...,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,0
2,39421,H1_45,1159,45,0,0,2,19.051092,0,0,...,0.0,0,1.0,0,0.0,0.0,0.0,0.0,40,40
3,39421,H2_15,1159,60,0,0,0,0.000000,0,0,...,0.0,0,1.0,0,0.0,0.0,0.0,0.0,0,40
4,39421,H2_30,1159,75,0,0,0,0.000000,0,0,...,0.0,0,1.0,0,0.0,0.0,0.0,0.0,0,40


In [5]:
sequence_feature_export = sequence_feature_table.drop(
    columns=['_seq15m_max_event_cont_min', '_seqpos_max_event_cont_min']
)
modeling_table = base.merge(
    sequence_feature_export,
    on=['player_appearance_id', 'checkpoint', 'fixture_id', 'checkpoint_cont_min'],
    how='left',
    validate='one_to_one',
)

validation_checks = pd.DataFrame(
    [
        {
            'check_name': 'seq15m_future_leak_rows',
            'value': int(
                (
                    sequence_feature_table['_seq15m_max_event_cont_min']
                    > sequence_feature_table['checkpoint_cont_min']
                ).sum()
            ),
            'expected': 0,
        },
        {
            'check_name': 'seqpos_future_leak_rows',
            'value': int(
                (
                    sequence_feature_table['_seqpos_max_event_cont_min']
                    > sequence_feature_table['checkpoint_cont_min']
                ).sum()
            ),
            'expected': 0,
        },
        {
            'check_name': 'join_row_delta',
            'value': int(len(modeling_table) - len(base)),
            'expected': 0,
        },
        {
            'check_name': 'missing_sequence_rows_after_join',
            'value': int(modeling_table['seq15m_pressure_total'].isna().sum()),
            'expected': 0,
        },
        {
            'check_name': 'ambiguous_pressure_rows',
            'value': int((pressure['possession_nunique'].fillna(0) > 1).sum()),
            'expected': 'diagnostic',
        },
        {
            'check_name': 'ambiguous_pressure_rows_linked',
            'value': int(
                ((pressure['possession_nunique'].fillna(0) > 1) & pressure['linked_pressure_flag'].eq(1)).sum()
            ),
            'expected': 0,
        },
    ]
)
validation_checks['status'] = np.where(
    validation_checks['expected'].astype(str) == 'diagnostic',
    'diagnostic',
    np.where(validation_checks['value'] == validation_checks['expected'], 'pass', 'fail'),
)

sequence_feature_columns = [
    column
    for column in sequence_feature_export.columns
    if column.startswith('seq15m_') or column.startswith('seqpos_')
]
feature_support = pd.DataFrame(
    [
        {
            'feature': column,
            'family': 'seq15m' if column.startswith('seq15m_') else 'seqpos',
            'nonzero_rate': float((sequence_feature_export[column] != 0).mean()),
            'mean': float(sequence_feature_export[column].mean()),
            'std': float(sequence_feature_export[column].std(ddof=0)),
        }
        for column in sequence_feature_columns
    ]
).sort_values(['family', 'nonzero_rate', 'feature'], ascending=[True, False, True]).reset_index(drop=True)

selected_trimmed_features = set(TRIMMED_SEQ15M_FEATURES + TRIMMED_SEQPOS_FEATURES)
trimmed_feature_selection = feature_support.copy()
trimmed_feature_selection['selected_for_trimmed_model'] = trimmed_feature_selection['feature'].isin(selected_trimmed_features)
trimmed_feature_selection['selection_reason'] = np.where(
    trimmed_feature_selection['selected_for_trimmed_model'],
    'supported_core',
    np.where(
        trimmed_feature_selection['nonzero_rate'] < 0.25,
        'dropped_low_support',
        'dropped_for_redundancy_or_instability',
    ),
)

selected_lean_features = set(LEAN_SEQ15M_FEATURES + LEAN_SEQPOS_FEATURES)
lean_feature_selection = trimmed_feature_selection.copy()
lean_feature_selection['selected_for_lean_model'] = lean_feature_selection['feature'].isin(selected_lean_features)
lean_feature_selection['lean_selection_reason'] = np.where(
    lean_feature_selection['selected_for_lean_model'],
    'kept_after_redundancy_pruning',
    np.where(
        lean_feature_selection['selected_for_trimmed_model'],
        'dropped_as_redundant_with_stronger_pairmate',
        lean_feature_selection['selection_reason'],
    ),
)

display(Markdown('### Validation Checks'))
display(validation_checks)
display(Markdown('### Feature Support Snapshot'))
display(feature_support.head(15))
display(Markdown('### Trimmed Supported-Core Selection'))
display(
    trimmed_feature_selection[
        ['feature', 'family', 'nonzero_rate', 'selected_for_trimmed_model', 'selection_reason']
    ].sort_values(['selected_for_trimmed_model', 'family', 'nonzero_rate'], ascending=[False, True, False]).reset_index(drop=True)
)
display(Markdown('### Lean Redundancy-Pruned Selection'))
display(
    lean_feature_selection[
        ['feature', 'family', 'nonzero_rate', 'selected_for_trimmed_model', 'selected_for_lean_model', 'lean_selection_reason']
    ].sort_values(['selected_for_lean_model', 'selected_for_trimmed_model', 'family', 'nonzero_rate'], ascending=[False, False, True, False]).reset_index(drop=True)
)

### Validation Checks

,check_name,value,expected,status
0,seq15m_future_leak_rows,0,0,pass
1,seqpos_future_leak_rows,0,0,pass
2,join_row_delta,0,0,pass
3,missing_sequence_rows_after_join,0,0,pass
4,ambiguous_pressure_rows,919,diagnostic,diagnostic
5,ambiguous_pressure_rows_linked,0,0,pass


### Feature Support Snapshot

,feature,family,nonzero_rate,mean,std
0,seq15m_active_minute_share,seq15m,0.946644,0.422872,0.198270
1,seq15m_longest_active_streak,seq15m,0.946644,2.813827,1.668631
2,seq15m_recency_weighted_run_total,seq15m,0.930293,4.117499,2.583137
3,seq15m_run_distance_total,seq15m,0.930293,367.125082,978.629056
4,seq15m_run_total,seq15m,0.930293,7.938325,4.798933
5,seq15m_run_intensity_slope,seq15m,0.930006,-0.650161,6.366789
6,seq15m_pressure_total,seq15m,0.820138,2.915089,2.614277
7,seq15m_recency_weighted_pressure_total,seq15m,0.820138,1.507669,1.458507
8,seq15m_pressure_density_slope,seq15m,0.812966,-0.002520,0.030454
9,seq15m_angle_concentration,seq15m,0.660929,0.577392,0.436994


### Trimmed Supported-Core Selection

,feature,family,nonzero_rate,selected_for_trimmed_model,selection_reason
0,seq15m_active_minute_share,seq15m,0.946644,True,supported_core
1,seq15m_run_intensity_slope,seq15m,0.930006,True,supported_core
2,seq15m_pressure_total,seq15m,0.820138,True,supported_core
3,seq15m_recency_weighted_pressure_total,seq15m,0.820138,True,supported_core
4,seq15m_angle_concentration,seq15m,0.660929,True,supported_core
5,seq15m_pressure_angle_availability_rate,seq15m,0.660929,True,supported_core
6,seq15m_pressured_minutes_with_runs_share,seq15m,0.635972,True,supported_core
7,seq15m_late_burst_count,seq15m,0.594664,True,supported_core
8,seq15m_pressure_turnover_total,seq15m,0.587780,True,supported_core
9,seq15m_lateral_angle_share,seq15m,0.537292,True,supported_core


### Lean Redundancy-Pruned Selection

,feature,family,nonzero_rate,selected_for_trimmed_model,selected_for_lean_model,lean_selection_reason
0,seq15m_active_minute_share,seq15m,0.946644,True,True,kept_after_redundancy_pruning
1,seq15m_pressure_total,seq15m,0.820138,True,True,kept_after_redundancy_pruning
2,seq15m_angle_concentration,seq15m,0.660929,True,True,kept_after_redundancy_pruning
3,seq15m_late_burst_count,seq15m,0.594664,True,True,kept_after_redundancy_pruning
4,seq15m_pressure_turnover_total,seq15m,0.587780,True,True,kept_after_redundancy_pruning
5,seqpos_mean_run_distance_per_possession,seqpos,0.964716,True,True,kept_after_redundancy_pruning
6,seqpos_mean_runs_per_possession,seqpos,0.964716,True,True,kept_after_redundancy_pruning
7,seqpos_possession_count,seqpos,0.964716,True,True,kept_after_redundancy_pruning
8,seqpos_unmatched_pressure_share,seqpos,0.869765,True,True,kept_after_redundancy_pruning
9,seqpos_linked_pressure_total,seqpos,0.794607,True,True,kept_after_redundancy_pruning


In [6]:
development_df = modeling_table[modeling_table['partition_role'] == 'development'].copy().reset_index(drop=True)

feature_sets = {
    'base_only': [],
    'base_plus_minute_hybrid': [column for column in modeling_table.columns if column.startswith('seq15m_')],
    'base_plus_possession_sequence': [column for column in modeling_table.columns if column.startswith('seqpos_')],
    'base_plus_both': [
        column
        for column in modeling_table.columns
        if column.startswith('seq15m_') or column.startswith('seqpos_')
    ],
    'base_plus_minute_hybrid_trimmed': TRIMMED_SEQ15M_FEATURES,
    'base_plus_possession_sequence_trimmed': TRIMMED_SEQPOS_FEATURES,
    'base_plus_both_trimmed': TRIMMED_SEQ15M_FEATURES + TRIMMED_SEQPOS_FEATURES,
    'base_plus_minute_hybrid_leaner': LEAN_SEQ15M_FEATURES,
    'base_plus_possession_sequence_leaner': LEAN_SEQPOS_FEATURES,
    'base_plus_both_leaner': LEAN_SEQ15M_FEATURES + LEAN_SEQPOS_FEATURES,
}

metrics_rows = []
fold_metric_rows = []
oof_registry = {}

for model_name in ['logreg', 'rf']:
    for feature_set_name, extra_columns in feature_sets.items():
        feature_columns = BASE_FEATURE_COLUMNS + extra_columns
        fold_metrics_df, oof_df = evaluate_feature_set(
            development_df=development_df,
            feature_columns=feature_columns,
            model_name=model_name,
            return_oof=(model_name == 'logreg' and feature_set_name == 'base_plus_both_leaner'),
        )
        summary_row = {
            'model_name': model_name,
            'feature_set': feature_set_name,
            'n_features': len(feature_columns),
        }
        for metric_name in ['balanced_accuracy', 'roc_auc', 'pr_auc', 'brier_score']:
            summary_row[metric_name] = float(fold_metrics_df[metric_name].mean())
        metrics_rows.append(summary_row)

        fold_metrics_export = fold_metrics_df.copy()
        fold_metrics_export.insert(0, 'model_name', model_name)
        fold_metrics_export.insert(1, 'feature_set', feature_set_name)
        fold_metric_rows.append(fold_metrics_export)

        if oof_df is not None:
            oof_registry[(model_name, feature_set_name)] = oof_df.copy()

metrics_table = pd.DataFrame(metrics_rows).sort_values(
    ['balanced_accuracy', 'roc_auc', 'pr_auc'],
    ascending=[False, False, False],
).reset_index(drop=True)
fold_metrics_table = pd.concat(fold_metric_rows, ignore_index=True)

display(Markdown('### Ablation Metrics'))
display(metrics_table)

### Ablation Metrics

,model_name,feature_set,n_features,balanced_accuracy,roc_auc,pr_auc,brier_score
0,logreg,base_only,25,0.583135,0.614308,0.094706,0.221995
1,logreg,base_plus_possession_sequence_leaner,30,0.580645,0.595758,0.091353,0.215797
2,logreg,base_plus_minute_hybrid_leaner,30,0.575027,0.614787,0.093967,0.219731
3,logreg,base_plus_minute_hybrid_trimmed,35,0.571059,0.609236,0.095369,0.219291
4,logreg,base_plus_both_leaner,35,0.568133,0.597268,0.092828,0.213876
5,logreg,base_plus_possession_sequence_trimmed,35,0.564746,0.592677,0.090828,0.214892
6,logreg,base_plus_minute_hybrid,51,0.562430,0.599728,0.096606,0.219781
7,logreg,base_plus_both_trimmed,45,0.562250,0.586414,0.089673,0.214087
8,logreg,base_plus_both,67,0.560841,0.566554,0.085662,0.215815
9,logreg,base_plus_possession_sequence,41,0.560769,0.581339,0.082633,0.215798


In [7]:
tuned_possession_feature_set_name = 'base_plus_possession_sequence_leaner'
tuned_possession_feature_columns = BASE_FEATURE_COLUMNS + LEAN_SEQPOS_FEATURES
tuned_possession_grid = [
    {'penalty': penalty, 'C': c_value, 'class_weight': class_weight}
    for penalty in ['l1', 'l2']
    for c_value in [0.03, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
    for class_weight in [None, 'balanced']
]

tuned_possession_rows = []
for config in tuned_possession_grid:
    fold_rows = []
    for fold in sorted(development_df['fold'].dropna().unique()):
        train_df = development_df[development_df['fold'] != fold].copy()
        valid_df = development_df[development_df['fold'] == fold].copy()
        tuned_pipeline = Pipeline(
            steps=[
                ('prep', build_preprocessor(tuned_possession_feature_columns)),
                (
                    'model',
                    LogisticRegression(
                        max_iter=5000,
                        solver='liblinear',
                        random_state=RANDOM_STATE,
                        penalty=config['penalty'],
                        C=config['C'],
                        class_weight=config['class_weight'],
                    ),
                ),
            ]
        )
        tuned_pipeline.fit(train_df[tuned_possession_feature_columns], train_df['scored_after'].astype(int))
        pred_prob = tuned_pipeline.predict_proba(valid_df[tuned_possession_feature_columns])[:, 1]
        fold_metrics = score_predictions(valid_df['scored_after'].to_numpy(), pred_prob)
        fold_rows.append(fold_metrics)
    fold_df = pd.DataFrame(fold_rows)
    tuned_possession_rows.append(
        {
            **config,
            **{metric_name: float(fold_df[metric_name].mean()) for metric_name in fold_df.columns},
        }
    )

tuned_possession_tuning_table = pd.DataFrame(tuned_possession_rows).sort_values(
    ['balanced_accuracy', 'roc_auc', 'pr_auc', 'brier_score'],
    ascending=[False, False, False, True],
).reset_index(drop=True)
best_tuned_possession_config = tuned_possession_tuning_table.iloc[0].to_dict()

tuned_possession_oof_parts = []
for fold in sorted(development_df['fold'].dropna().unique()):
    train_df = development_df[development_df['fold'] != fold].copy()
    valid_df = development_df[development_df['fold'] == fold].copy()
    tuned_pipeline = Pipeline(
        steps=[
            ('prep', build_preprocessor(tuned_possession_feature_columns)),
            (
                'model',
                LogisticRegression(
                    max_iter=5000,
                    solver='liblinear',
                    random_state=RANDOM_STATE,
                    penalty=best_tuned_possession_config['penalty'],
                    C=float(best_tuned_possession_config['C']),
                    class_weight=best_tuned_possession_config['class_weight'],
                ),
            ),
        ]
    )
    tuned_pipeline.fit(train_df[tuned_possession_feature_columns], train_df['scored_after'].astype(int))
    pred_prob = tuned_pipeline.predict_proba(valid_df[tuned_possession_feature_columns])[:, 1]
    oof_part = valid_df[
        ['player_appearance_id', 'fixture_id', 'checkpoint', 'position', 'scored_after']
    ].copy()
    oof_part['fold'] = fold
    oof_part['pred_prob'] = pred_prob
    tuned_possession_oof_parts.append(oof_part)

tuned_possession_oof_predictions = pd.concat(tuned_possession_oof_parts, ignore_index=True)
tuned_possession_final_pipeline = Pipeline(
    steps=[
        ('prep', build_preprocessor(tuned_possession_feature_columns)),
        (
            'model',
            LogisticRegression(
                max_iter=5000,
                solver='liblinear',
                random_state=RANDOM_STATE,
                penalty=best_tuned_possession_config['penalty'],
                C=float(best_tuned_possession_config['C']),
                class_weight=best_tuned_possession_config['class_weight'],
            ),
        ),
    ]
)
tuned_possession_final_pipeline.fit(
    development_df[tuned_possession_feature_columns],
    development_df['scored_after'].astype(int),
)
tuned_possession_coef_frame = pd.DataFrame(
    {
        'feature': tuned_possession_final_pipeline.named_steps['prep'].get_feature_names_out(),
        'coefficient': tuned_possession_final_pipeline.named_steps['model'].coef_[0],
    }
)
tuned_possession_coef_frame['feature'] = tuned_possession_coef_frame['feature'].str.replace('num__', '', regex=False).str.replace('cat__', '', regex=False)
tuned_possession_coef_frame['abs_coefficient'] = tuned_possession_coef_frame['coefficient'].abs()
tuned_possession_interpretation_table = tuned_possession_coef_frame[
    tuned_possession_coef_frame['feature'].isin(LEAN_SEQPOS_FEATURES)
].sort_values('abs_coefficient', ascending=False).reset_index(drop=True)
tuned_possession_subgroup_stability_table = pd.concat(
    [
        subgroup_metric_table(tuned_possession_oof_predictions, 'checkpoint'),
        subgroup_metric_table(tuned_possession_oof_predictions, 'position'),
    ],
    ignore_index=True,
)

display(Markdown('### Lean Possession-Only Logistic Tuning'))
display(tuned_possession_tuning_table.head(12))
display(Markdown(f"Best config: `penalty={best_tuned_possession_config['penalty']}`, `C={float(best_tuned_possession_config['C']):.2f}`, `class_weight={best_tuned_possession_config['class_weight']}`"))
display(Markdown('### Tuned Possession-Only Effects'))
display(tuned_possession_interpretation_table)
display(Markdown('### Subgroup Stability For Tuned Possession-Only Logistic'))
display(tuned_possession_subgroup_stability_table)

rq2_behavior_labels = {
    'seqpos_mean_run_distance_per_possession': 'Possession run-distance intensity',
    'seqpos_mean_runs_per_possession': 'Runs per possession',
    'seqpos_possession_count': 'Count of distinct possessions before the checkpoint',
    'seqpos_linked_pressure_total': 'Strictly linked under-pressure actions inside possessions',
    'seqpos_unmatched_pressure_share': 'Residual share of pressure events that do not map cleanly to possessions',
}
tuned_possession_transformed = tuned_possession_final_pipeline.named_steps['prep'].transform(
    development_df[tuned_possession_feature_columns]
)
if hasattr(tuned_possession_transformed, 'toarray'):
    tuned_possession_transformed = tuned_possession_transformed.toarray()

tuned_feature_names = pd.Index(
    tuned_possession_final_pipeline.named_steps['prep'].get_feature_names_out()
)
tuned_seqpos_feature_names = [f'num__{feature}' for feature in LEAN_SEQPOS_FEATURES]
tuned_seqpos_indices = [tuned_feature_names.get_loc(feature_name) for feature_name in tuned_seqpos_feature_names]
tuned_seqpos_raw = development_df[LEAN_SEQPOS_FEATURES].reset_index(drop=True)
tuned_shap_explainer = shap.LinearExplainer(
    tuned_possession_final_pipeline.named_steps['model'],
    tuned_possession_transformed,
)
tuned_shap_values_full = tuned_shap_explainer.shap_values(tuned_possession_transformed)
tuned_shap_values_seqpos = tuned_shap_values_full[:, tuned_seqpos_indices]

tuned_possession_shap_table = pd.DataFrame(
    {
        'feature': LEAN_SEQPOS_FEATURES,
        'behavioral_aspect': [rq2_behavior_labels[feature] for feature in LEAN_SEQPOS_FEATURES],
        'mean_abs_shap': np.abs(tuned_shap_values_seqpos).mean(axis=0),
        'mean_shap': tuned_shap_values_seqpos.mean(axis=0),
        'feature_mean': tuned_seqpos_raw.mean().to_numpy(),
        'feature_std': tuned_seqpos_raw.std(ddof=0).to_numpy(),
        'coefficient': tuned_possession_interpretation_table.set_index('feature').reindex(LEAN_SEQPOS_FEATURES)['coefficient'].to_numpy(),
        'nonzero_rate': [(tuned_seqpos_raw[feature] != 0).mean() for feature in LEAN_SEQPOS_FEATURES],
    }
)
tuned_possession_shap_table['feature_value_shap_corr'] = [
    np.corrcoef(tuned_seqpos_raw[feature], tuned_shap_values_seqpos[:, idx])[0, 1]
    if tuned_seqpos_raw[feature].nunique() > 1
    else np.nan
    for idx, feature in enumerate(LEAN_SEQPOS_FEATURES)
]
tuned_possession_shap_table['effect_direction'] = np.where(
    tuned_possession_shap_table['coefficient'] > 0.02,
    'higher_feature -> higher_goal_probability',
    np.where(
        tuned_possession_shap_table['coefficient'] < -0.02,
        'higher_feature -> lower_goal_probability',
        'near_zero_net_effect',
    ),
)
tuned_possession_shap_table['effect_strength'] = np.where(
    tuned_possession_shap_table['coefficient'].abs() >= 0.20,
    'strong',
    np.where(
        tuned_possession_shap_table['coefficient'].abs() >= 0.05,
        'moderate',
        np.where(tuned_possession_shap_table['coefficient'].abs() > 0.0, 'weak', 'near_zero'),
    ),
)
tuned_possession_shap_table['rq2_takeaway'] = [
    'Longer average run distance inside possessions is the clearest positive scoring signal.'
    if feature == 'seqpos_mean_run_distance_per_possession'
    else 'Higher unmatched-pressure share is the clearest negative behavioural signal.'
    if feature == 'seqpos_unmatched_pressure_share'
    else 'More linked pressure actions inside possessions weakly increase later scoring probability.'
    if feature == 'seqpos_linked_pressure_total'
    else 'More distinct possessions before the checkpoint weakly increase later scoring probability.'
    if feature == 'seqpos_possession_count'
    else 'Runs per possession contributes little once the tuned sparse model selects its core effects.'
    for feature in LEAN_SEQPOS_FEATURES
]
tuned_possession_shap_table = tuned_possession_shap_table.sort_values(
    ['mean_abs_shap', 'coefficient'],
    ascending=[False, False],
).reset_index(drop=True)
tuned_possession_shap_table.insert(0, 'importance_rank', np.arange(1, len(tuned_possession_shap_table) + 1))

plt.figure(figsize=(9, 4.5))
shap.summary_plot(
    tuned_shap_values_seqpos,
    features=tuned_seqpos_raw,
    feature_names=LEAN_SEQPOS_FEATURES,
    plot_type='bar',
    show=False,
)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'tuned_possession_shap_bar.png', dpi=200, bbox_inches='tight')
plt.close()

plt.figure(figsize=(9, 5.5))
shap.summary_plot(
    tuned_shap_values_seqpos,
    features=tuned_seqpos_raw,
    feature_names=LEAN_SEQPOS_FEATURES,
    show=False,
)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'tuned_possession_shap_beeswarm.png', dpi=200, bbox_inches='tight')
plt.close()

display(Markdown('### RQ2: Behavioural Determinants From Tuned Possession-Only SHAP'))
display(tuned_possession_shap_table)
display(Markdown('SHAP plots were saved as `tuned_possession_shap_bar.png` and `tuned_possession_shap_beeswarm.png` in `artifacts/sequential_goal_forecasting_geometry/`.'))


### Lean Possession-Only Logistic Tuning

,penalty,C,class_weight,balanced_accuracy,pr_auc,brier_score,roc_auc
0,l1,0.10,balanced,0.600574,0.097527,0.214964,0.613030
1,l1,0.25,balanced,0.593328,0.093189,0.215169,0.608629
2,l1,0.03,balanced,0.586786,0.091998,0.233454,0.605526
3,l1,0.50,balanced,0.585560,0.092191,0.216563,0.603174
4,l2,0.50,balanced,0.583067,0.090947,0.216413,0.593871
5,l2,1.00,balanced,0.582921,0.090979,0.216994,0.592552
6,l2,0.25,balanced,0.580645,0.091353,0.215797,0.595758
7,l2,2.00,balanced,0.580351,0.090015,0.217395,0.591015
8,l1,1.00,balanced,0.576722,0.090744,0.217610,0.597651
9,l2,4.00,balanced,0.575220,0.089332,0.217557,0.590045


Best config: `penalty=l1`, `C=0.10`, `class_weight=balanced`

### Tuned Possession-Only Effects

,feature,coefficient,abs_coefficient
0,seqpos_mean_run_distance_per_possession,0.603079,0.603079
1,seqpos_unmatched_pressure_share,-0.193912,0.193912
2,seqpos_linked_pressure_total,0.049954,0.049954
3,seqpos_possession_count,0.034973,0.034973
4,seqpos_mean_runs_per_possession,0.000000,0.000000


### Subgroup Stability For Tuned Possession-Only Logistic

,group_type,group_value,rows,positive_rate,balanced_accuracy,pr_auc,brier_score,roc_auc
0,checkpoint,H1_15,549,0.087432,0.587138,0.114627,0.293043,0.595351
1,checkpoint,H1_30,549,0.067395,0.583272,0.095018,0.254790,0.589052
2,checkpoint,H1_45,549,0.058288,0.544669,0.066528,0.219091,0.537355
3,checkpoint,H2_15,549,0.041894,0.565548,0.098571,0.165203,0.543561
4,checkpoint,H2_30,549,0.040073,0.619113,0.077138,0.158806,0.600483
5,checkpoint,H2_45,43,0.046512,0.689024,0.170455,0.121424,0.719512
6,checkpoint,ET1_15,42,0.023810,0.451220,0.038462,0.114144,0.390244
7,position,M,1050,0.069524,0.493053,0.059429,0.262250,0.440179
8,position,D,1009,0.028741,0.476425,0.022423,0.125742,0.352111
9,position,A,513,0.122807,0.532063,0.125735,0.392228,0.518448


### RQ2: Behavioural Determinants From Tuned Possession-Only SHAP

,importance_rank,feature,behavioral_aspect,mean_abs_shap,mean_shap,feature_mean,feature_std,coefficient,nonzero_rate,feature_value_shap_corr,effect_direction,effect_strength,rq2_takeaway
0,1,seqpos_mean_run_distance_per_possession,Possession run-distance intensity,0.371392,-0.083997,573.515933,1493.884331,0.603079,0.964311,1.0,higher_feature -> higher_goal_probability,strong,Longer average run distance inside possessions...
1,2,seqpos_unmatched_pressure_share,Residual share of pressure events that do not ...,0.155930,0.004989,0.527357,0.298928,-0.193912,0.868905,-1.0,higher_feature -> lower_goal_probability,moderate,Higher unmatched-pressure share is the cleares...
2,3,seqpos_linked_pressure_total,Strictly linked under-pressure actions inside ...,0.039680,-0.001389,3.952650,3.860997,0.049954,0.799293,1.0,higher_feature -> higher_goal_probability,weak,More linked pressure actions inside possession...
3,4,seqpos_possession_count,Count of distinct possessions before the check...,0.018382,-0.003047,1.851237,0.444984,0.034973,0.964311,1.0,higher_feature -> higher_goal_probability,weak,More distinct possessions before the checkpoin...
4,5,seqpos_mean_runs_per_possession,Runs per possession,0.000000,0.000000,11.520318,8.057108,0.000000,0.964311,NaN,near_zero_net_effect,near_zero,Runs per possession contributes little once th...


SHAP plots were saved as `tuned_possession_shap_bar.png` and `tuned_possession_shap_beeswarm.png` in `artifacts/sequential_goal_forecasting_geometry/`.

In [8]:
best_feature_set_name = 'base_plus_both_leaner'
best_logreg_feature_columns = BASE_FEATURE_COLUMNS + feature_sets[best_feature_set_name]
interpretation_pipeline = Pipeline(
    steps=[
        ('prep', build_preprocessor(best_logreg_feature_columns)),
        (
            'model',
            LogisticRegression(
                max_iter=4000,
                class_weight='balanced',
                solver='liblinear',
                C=0.25,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
interpretation_pipeline.fit(
    development_df[best_logreg_feature_columns],
    development_df['scored_after'].astype(int),
)
coefficient_frame = pd.DataFrame(
    {
        'feature': interpretation_pipeline.named_steps['prep'].get_feature_names_out(),
        'coefficient': interpretation_pipeline.named_steps['model'].coef_[0],
    }
)
coefficient_frame['feature'] = coefficient_frame['feature'].str.replace('num__', '', regex=False).str.replace('cat__', '', regex=False)
coefficient_frame['abs_coefficient'] = coefficient_frame['coefficient'].abs()
interpretation_table = coefficient_frame[
    coefficient_frame['feature'].str.startswith('seq15m_')
    | coefficient_frame['feature'].str.startswith('seqpos_')
].sort_values('abs_coefficient', ascending=False).head(20).reset_index(drop=True)

subgroup_predictions = oof_registry[('logreg', best_feature_set_name)].copy()
subgroup_stability_table = pd.concat(
    [
        subgroup_metric_table(subgroup_predictions, 'checkpoint'),
        subgroup_metric_table(subgroup_predictions, 'position'),
    ],
    ignore_index=True,
)

display(Markdown('### Secondary Lean Combined Effects'))
display(interpretation_table)
display(Markdown(f'### Secondary Subgroup Stability For `logreg + {best_feature_set_name}`'))
display(subgroup_stability_table)

### Secondary Lean Combined Effects

,feature,coefficient,abs_coefficient
0,seqpos_mean_run_distance_per_possession,1.382633,1.382633
1,seq15m_pressure_total,-0.480689,0.480689
2,seq15m_pressure_turnover_total,0.308846,0.308846
3,seqpos_mean_runs_per_possession,0.271943,0.271943
4,seqpos_possession_count,0.239808,0.239808
5,seqpos_unmatched_pressure_share,-0.202591,0.202591
6,seqpos_linked_pressure_total,0.180363,0.180363
7,seq15m_active_minute_share,0.164034,0.164034
8,seq15m_angle_concentration,0.116977,0.116977
9,seq15m_late_burst_count,0.096099,0.096099


### Secondary Subgroup Stability For `logreg + base_plus_both_leaner`

,group_type,group_value,rows,positive_rate,balanced_accuracy,pr_auc,brier_score,roc_auc
0,checkpoint,H1_15,549,0.087432,0.556013,0.121272,0.293910,0.608283
1,checkpoint,H1_30,549,0.067395,0.574958,0.092657,0.241139,0.582454
2,checkpoint,H1_45,549,0.058288,0.507616,0.058377,0.233841,0.507797
3,checkpoint,H2_15,549,0.041894,0.546661,0.059257,0.160429,0.524467
4,checkpoint,H2_30,549,0.040073,0.585044,0.080527,0.149667,0.593928
5,checkpoint,H2_45,43,0.046512,0.390244,0.051151,0.175107,0.341463
6,checkpoint,ET1_15,42,0.023810,0.414634,0.027027,0.145822,0.121951
7,position,M,1050,0.069524,0.463531,0.062993,0.262424,0.441777
8,position,D,1009,0.028741,0.467752,0.022357,0.119265,0.373610
9,position,A,513,0.122807,0.517619,0.120834,0.402423,0.503527


In [9]:
sequence_feature_export.to_csv(ARTIFACT_DIR / 'sequence_feature_table.csv', index=False)
minute_sequence_long.to_csv(ARTIFACT_DIR / 'minute_sequence_long.csv', index=False)
possession_sequence_long.to_csv(ARTIFACT_DIR / 'possession_sequence_long.csv', index=False)
validation_checks.to_csv(ARTIFACT_DIR / 'validation_checks.csv', index=False)
feature_support.to_csv(ARTIFACT_DIR / 'feature_support.csv', index=False)
trimmed_feature_selection.to_csv(ARTIFACT_DIR / 'trimmed_feature_selection.csv', index=False)
lean_feature_selection.to_csv(ARTIFACT_DIR / 'lean_feature_selection.csv', index=False)
tuned_possession_tuning_table.to_csv(ARTIFACT_DIR / 'tuned_possession_tuning_table.csv', index=False)
tuned_possession_oof_predictions.to_csv(ARTIFACT_DIR / 'tuned_possession_oof_predictions.csv', index=False)
tuned_possession_interpretation_table.to_csv(ARTIFACT_DIR / 'tuned_possession_interpretation_table.csv', index=False)
tuned_possession_subgroup_stability_table.to_csv(ARTIFACT_DIR / 'tuned_possession_subgroup_stability_table.csv', index=False)
tuned_possession_shap_table.to_csv(ARTIFACT_DIR / 'tuned_possession_shap_table.csv', index=False)
metrics_table.to_csv(ARTIFACT_DIR / 'metrics_table.csv', index=False)
fold_metrics_table.to_csv(ARTIFACT_DIR / 'fold_metrics_table.csv', index=False)
interpretation_table.to_csv(ARTIFACT_DIR / 'interpretation_table.csv', index=False)
subgroup_stability_table.to_csv(ARTIFACT_DIR / 'subgroup_stability_table.csv', index=False)

display(Markdown('### Exported Artifacts'))
display(
    pd.DataFrame(
        {
            'artifact': [
                'sequence_feature_table.csv',
                'minute_sequence_long.csv',
                'possession_sequence_long.csv',
                'validation_checks.csv',
                'feature_support.csv',
                'trimmed_feature_selection.csv',
                'lean_feature_selection.csv',
                'tuned_possession_tuning_table.csv',
                'tuned_possession_oof_predictions.csv',
                'tuned_possession_interpretation_table.csv',
                'tuned_possession_subgroup_stability_table.csv',
                'tuned_possession_shap_table.csv',
                'tuned_possession_shap_bar.png',
                'tuned_possession_shap_beeswarm.png',
                'metrics_table.csv',
                'fold_metrics_table.csv',
                'interpretation_table.csv',
                'subgroup_stability_table.csv',
            ],
            'path': [str(ARTIFACT_DIR / name) for name in [
                'sequence_feature_table.csv',
                'minute_sequence_long.csv',
                'possession_sequence_long.csv',
                'validation_checks.csv',
                'feature_support.csv',
                'trimmed_feature_selection.csv',
                'lean_feature_selection.csv',
                'tuned_possession_tuning_table.csv',
                'tuned_possession_oof_predictions.csv',
                'tuned_possession_interpretation_table.csv',
                'tuned_possession_subgroup_stability_table.csv',
                'tuned_possession_shap_table.csv',
                'tuned_possession_shap_bar.png',
                'tuned_possession_shap_beeswarm.png',
                'metrics_table.csv',
                'fold_metrics_table.csv',
                'interpretation_table.csv',
                'subgroup_stability_table.csv',
            ]],
        }
    )
)

### Exported Artifacts

,artifact,path
0,sequence_feature_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
1,minute_sequence_long.csv,/Users/jan/Documents/competitions/hackatons/WE...
2,possession_sequence_long.csv,/Users/jan/Documents/competitions/hackatons/WE...
3,validation_checks.csv,/Users/jan/Documents/competitions/hackatons/WE...
4,feature_support.csv,/Users/jan/Documents/competitions/hackatons/WE...
5,trimmed_feature_selection.csv,/Users/jan/Documents/competitions/hackatons/WE...
6,lean_feature_selection.csv,/Users/jan/Documents/competitions/hackatons/WE...
7,tuned_possession_tuning_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
8,tuned_possession_oof_predictions.csv,/Users/jan/Documents/competitions/hackatons/WE...
9,tuned_possession_interpretation_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
